# Assignment 7

**Course:** COMP 7712 - Algorithms

**Due Date:** 4/28/2026, 9:00 AM

**Topics:** Backtracking, Dynamic Programming, Memoization

**Instructions:**
- This assignment has **three parts**, all organized around a single problem: *rod cutting*.
- You will solve the same problem three ways: with **backtracking**, with a **dynamic-programming recurrence**, and with **memoization**.
- For coding problems, write your solution in the provided code cell. **Do not use built-in functions** (like `sum`, `max`, `min`, `sorted`) unless explicitly allowed.
- **Do not modify the `backtrack` template** in Part 1 — only supply the knobs (`solution`, `possibilities`, `is_feasible`, `display_solution`).
- Make sure your code runs without errors before submitting.

---

## The Rod-Cutting Problem

You are given a rod of length $n$ (an integer) and a price table $p$ where $p[i]$ is the price you receive for a rod piece of length $i$, for $i = 1, 2, \ldots, n$. You may cut the rod into any number of integer-length pieces (including zero cuts, i.e. selling it whole) and sell each piece at its price. Your goal is to determine the maximum total revenue $r(n)$ obtainable from a rod of length $n$.

Throughout this assignment we use the following price table:

| length $i$ | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 |
|---|---|---|---|---|---|---|---|---|
| $p[i]$     | 1 | 5 | 8 | 9 | 10 | 17 | 17 | 20 |

```python
P = [0, 1, 5, 8, 9, 10, 17, 17, 20]
```

### A small worked example ($n=2$)

Before tackling the parts below, here is a tiny instance walked through end-to-end so the setup is concrete.

Picture a rod of length $2$ as two unit segments joined together:

```
[ 1 | 1 ]
```

There is exactly **one** place where you could cut it (the boundary between the two unit segments), so you only have two options:

| Cutting     | Pieces     | Revenue from price table          |
|-------------|------------|-----------------------------------|
| Don't cut   | $[2]$      | $p[2] = 5$                        |
| Cut once    | $[1, 1]$   | $p[1] + p[1] = 1 + 1 = 2$         |

The maximum revenue is $r(2) = 5$, achieved by selling the rod whole.

That is *all* the rod-cutting problem asks: list every way to break the rod into integer-length pieces, total up the prices for those pieces, and report the largest total. The three parts of this assignment will each compute that largest total a different way.

---
## The `backtrack` template (from Lecture 15)

We use the template **without modification**. Each problem below supplies only the knobs (`solution`, `possibilities`, `is_feasible`, `display_solution`).

In [8]:
def backtrack(solution, i, possibilities,
              is_feasible=lambda sol, i: True,
              display_solution=lambda sol: sol):
    if i == len(solution):
        print(display_solution(solution))
    else:
        for choice in possibilities:
            solution[i] = choice
            if is_feasible(solution, i):
                backtrack(solution, i + 1, possibilities,
                          is_feasible, display_solution)

---
## Part 1 — Backtracking

### Problem 1.1 — Hand enumeration

For a rod of length $4$, list every ordered cutting and its revenue.

**Answer.**

There are $2^{4-1} = 8$ ordered cuttings:

| Cutting | Revenue |
|---|---|
| $[4]$ | $p[4] = 9$ |
| $[1,3]$ | $p[1] + p[3] = 1 + 8 = 9$ |
| $[3,1]$ | $p[3] + p[1] = 8 + 1 = 9$ |
| $[2,2]$ | $p[2] + p[2] = 5 + 5 = \mathbf{10}$ |
| $[1,1,2]$ | $1 + 1 + 5 = 7$ |
| $[1,2,1]$ | $1 + 5 + 1 = 7$ |
| $[2,1,1]$ | $5 + 1 + 1 = 7$ |
| $[1,1,1,1]$ | $4 \cdot 1 = 4$ |

The unique maximum cutting is $[2,2]$, with $r(4) = 10$.

```
P = [0, 1, 5, 8, 9, 10, 17, 17, 20]
```

### Example — indicator lists on the $n=2$ rod

Before Problem 1.2 asks you to encode cuttings of a length-$4$ rod, here's what the indicator-list idea looks like on the length-$2$ rod from the worked example above.

A length-$2$ rod has $n - 1 = 1$ gap, so each cutting is described by an indicator list of length $1$:

| Indicator list | Meaning                          | Resulting cutting |
|----------------|----------------------------------|-------------------|
| `[False]`      | don't cut at the only gap        | $[2]$ (uncut)     |
| `[True]`       | cut at the only gap              | $[1, 1]$          |

In general, position $i$ in the indicator list describes what happens at gap $i$ (between unit segments $i$ and $i+1$). To recover the piece lengths from an indicator list, walk left-to-right: every `True` ends the current piece and starts a new one.

### Problem 1.2 — Represent cuttings as indicator lists

For a length-$n$ rod, `solution[i] = True` means "cut at gap $i$" (the gap between unit positions $i{+}1$ and $i{+}2$).

**Answer.**

For a length-$4$ rod the three gaps are between positions 1–2 (gap 0), 2–3 (gap 1), 3–4 (gap 2).

**(a)** Cutting $2+1+1$ means: don't cut gap 0 (so positions 1–2 stay together), cut gap 1 (isolating position 3 from the first piece), cut gap 2 (isolating position 4 from position 3). Indicator list: `[False, True, True]`.

**(b)** The uncut rod cuts at no gap, so the indicator list is `[False, False, False]`.

**(c)** `[True, True, True]` cuts at every gap, producing four pieces of length 1. Revenue = $4 \cdot p[1] = 4$.

### Problem 1.3 — Enumerate all cuttings using the template

Write `enumerate_cuttings(p, n)` that prints every cutting together with its revenue.

In [18]:
def enumerate_cuttings(p, n):
    solution = [None] * (n-1)
    possibilities = [False, True]

    def display_solution(sol):
        # Decode the indicator list into piece lengths.
        pieces, start = [], 0
        for i in range(len(sol)):
            if sol[i]:
                pieces.append(i + 1 - start)
                start = i + 1
        pieces.append(n - start)
        print('\t', sol, pieces)
        # Compute total revenue without using sum().
        revenue = 0
        for piece in pieces:
            revenue += p[piece]
        return (pieces, revenue)

    backtrack(solution, 0, possibilities, display_solution=display_solution)

In [22]:
# Try it on a length-3 rod. The four cuttings are [3], [2,1], [1,2], [1,1,1].
P = [0, 1, 5, 8, 9, 10, 17, 17, 20]
enumerate_cuttings(P, 4)

	 [False, False, False] [4]
([4], 9)
	 [False, False, True] [3, 1]
([3, 1], 9)
	 [False, True, False] [2, 2]
([2, 2], 10)
	 [False, True, True] [2, 1, 1]
([2, 1, 1], 7)
	 [True, False, False] [1, 3]
([1, 3], 9)
	 [True, False, True] [1, 2, 1]
([1, 2, 1], 7)
	 [True, True, False] [1, 1, 2]
([1, 1, 2], 7)
	 [True, True, True] [1, 1, 1, 1]
([1, 1, 1, 1], 4)


### Problem 1.4 — Find the maximum revenue using the template

Write `rod_cut_backtrack(p, n)` that returns $r(n)$.

In [9]:
def rod_cut_backtrack(p, n):
    if n == 0:
        return 0

    best = [0]                         # closure-captured running max
    solution = [None] * (n - 1)

    def visit(sol):
        # Decode the indicator list, compute revenue, update best.
        pieces, start = [], 0
        for i in range(len(sol)):
            if sol[i]:
                pieces.append(i + 1 - start)
                start = i + 1
        pieces.append(n - start)

        total = 0
        for piece in pieces:
            total += p[piece]

        if total > best[0]:
            best[0] = total
        return (pieces, total)         # template will print this

    backtrack(solution, 0, [False, True], display_solution=visit)
    return best[0]

In [10]:
# Sanity checks.
P = [0, 1, 5, 8, 9, 10, 17, 17, 20]
assert rod_cut_backtrack(P, 0) == 0
assert rod_cut_backtrack(P, 1) == 1
assert rod_cut_backtrack(P, 4) == 10
assert rod_cut_backtrack(P, 5) == 13
assert rod_cut_backtrack(P, 8) == 22
print("All checks passed.")

([1], 1)
([4], 9)
([3, 1], 9)
([2, 2], 10)
([2, 1, 1], 7)
([1, 3], 9)
([1, 2, 1], 7)
([1, 1, 2], 7)
([1, 1, 1, 1], 4)
([5], 10)
([4, 1], 10)
([3, 2], 13)
([3, 1, 1], 10)
([2, 3], 13)
([2, 2, 1], 11)
([2, 1, 2], 11)
([2, 1, 1, 1], 8)
([1, 4], 10)
([1, 3, 1], 10)
([1, 2, 2], 11)
([1, 2, 1, 1], 8)
([1, 1, 3], 10)
([1, 1, 2, 1], 8)
([1, 1, 1, 2], 8)
([1, 1, 1, 1, 1], 5)
([8], 20)
([7, 1], 18)
([6, 2], 22)
([6, 1, 1], 19)
([5, 3], 18)
([5, 2, 1], 16)
([5, 1, 2], 16)
([5, 1, 1, 1], 13)
([4, 4], 18)
([4, 3, 1], 18)
([4, 2, 2], 19)
([4, 2, 1, 1], 16)
([4, 1, 3], 18)
([4, 1, 2, 1], 16)
([4, 1, 1, 2], 16)
([4, 1, 1, 1, 1], 13)
([3, 5], 18)
([3, 4, 1], 18)
([3, 3, 2], 21)
([3, 3, 1, 1], 18)
([3, 2, 3], 21)
([3, 2, 2, 1], 19)
([3, 2, 1, 2], 19)
([3, 2, 1, 1, 1], 16)
([3, 1, 4], 18)
([3, 1, 3, 1], 18)
([3, 1, 2, 2], 19)
([3, 1, 2, 1, 1], 16)
([3, 1, 1, 3], 18)
([3, 1, 1, 2, 1], 16)
([3, 1, 1, 1, 2], 16)
([3, 1, 1, 1, 1, 1], 13)
([2, 6], 22)
([2, 5, 1], 16)
([2, 4, 2], 19)
([2, 4, 1, 1], 16)
([2, 3,

---
## Part 2 — Dynamic programming: the recurrence

### Problem 2.1 — First-choice recurrence


Think about cutting a rod of length $n$. Whatever you do, the **first piece** you cut off has some length $i \in \{1, 2, \ldots, n\}$. You earn $p[i]$ for that piece, and the remaining rod has length $n - i$ — a smaller instance of the *same problem*.

**(a)** Using the first-choice analysis above, write a recurrence for $r(n)$ in terms of $p$ and values of $r$ at smaller arguments. State the base case.

**(b)** In 2–3 sentences, justify the recurrence. What is the interpretation of the index you are taking a maximum over?

**Answer.**

**(a)** For $n \geq 1$,

$$r(n) = \max_{1 \leq i \leq n} \bigl\{\, p[i] + r(n - i) \,\bigr\},$$

with base case $r(0) = 0$.

**(b)** The index $i$ is the length of the **first piece** we cut off. Whatever the optimal cutting looks like, its leftmost piece has some integer length $i \in \{1, \ldots, n\}$; we collect $p[i]$ for that piece and are left with a rod of length $n - i$, which must itself be cut optimally — that is $r(n - i)$ by definition. The outer $\max$ simply picks the best first-piece length.

### Problem 2.2 — Implement the recurrence (no cache)

In [11]:
def rod_cut_dp(p, n):
    if n == 0:
        return 0
    best = p[n]                        # i = n: keep the whole rod
    for i in range(1, n):
        candidate = p[i] + rod_cut_dp(p, n - i)
        if candidate > best:
            best = candidate
    return best

In [19]:
# The recurrence must agree with Part 1's backtracking answer.
P = [0, 1, 5, 8, 9, 10, 17, 17, 20]

rod_cut_dp(P, 5)


13

### Observation — overlapping subproblems

Tracing `rod_cut_dp(P, 4)` shows that `rod_cut_dp(P, 2)` is computed from scratch several times (once via the $i=1$ branch, once via the $i=2$ branch of the top call, and again inside deeper calls). That's the overlapping-subproblems pattern Part 3 fixes.

---
## Part 3 — Memoization

### Problem 3.1 — Memoized rod cutting

In [13]:
def rod_cut_memo(p, n):
    cache = {}                         # fresh cache per top-level call

    def helper(m):
        if m in cache:
            return cache[m]
        if m == 0:
            cache[m] = 0
            return 0
        best = p[m]                    # i = m: keep the whole rod
        for i in range(1, m):
            candidate = p[i] + helper(m - i)
            if candidate > best:
                best = candidate
        cache[m] = best
        return best

    return helper(n)

In [14]:
# Must match rod_cut_dp on every input.
P = [0, 1, 5, 8, 9, 10, 17, 17, 20]
for k in range(len(P)):
    assert rod_cut_memo(P, k) == rod_cut_dp(P, k), f"Mismatch at n={k}"
print("All checks passed.")

All checks passed.


----

## Iterate `backtrack` in a loop

In [25]:
def backtrack(solution, 
              i, 
              possibilities,
              is_feasible=lambda sol, i: True,
              display_solution=lambda sol: sol):
    if i == len(solution):
        print(display_solution(solution))
    else:
        for choice in possibilities:
            solution[i] = choice
            if is_feasible(solution, i):
                backtrack(solution, i + 1, possibilities, is_feasible, display_solution)

In [31]:
backtrack([None]*3, 0, ['a','o'])

['a', 'a', 'a']
['a', 'a', 'o']
['a', 'o', 'a']
['a', 'o', 'o']
['o', 'a', 'a']
['o', 'a', 'o']
['o', 'o', 'a']
['o', 'o', 'o']


In [113]:
def backtrack(solution, 
              i, 
              possibilities,
              is_feasible=lambda sol, i: True,
              display_solution=lambda sol: sol):
    if i == len(solution):
        yield solution.copy()
    else:
        for choice in possibilities:
            solution[i] = choice
            if is_feasible(solution, i):
                backtrack(solution, i + 1, possibilities, is_feasible, display_solution)

In [115]:
for x in backtrack([None]*3, 0, ['a','o']):
    print(x)

In [117]:
list(backtrack([None]*3, 0, ['a','o']))


[]